In [18]:
import pandas as pd
import duckdb

athletes = pd.read_csv('./Data/athlete_events.csv')

athletes.info() # Obtenemos la informacion de los datos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  object 
 2   Sex     271116 non-null  object 
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  object 
 7   NOC     271116 non-null  object 
 8   Games   271116 non-null  object 
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  object 
 11  City    271116 non-null  object 
 12  Sport   271116 non-null  object 
 13  Event   271116 non-null  object 
 14  Medal   39783 non-null   object 
dtypes: float64(3), int64(2), object(10)
memory usage: 31.0+ MB


In [19]:
duckdb.query("""
select * 
from athletes
""").to_df()


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,None
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,None
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,None
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271111,135569,Andrzej ya,M,29.0,179.0,89.0,Poland-1,POL,1976 Winter,1976,Winter,Innsbruck,Luge,Luge Mixed (Men)'s Doubles,None
271112,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Individual",None
271113,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Team",None
271114,135571,Tomasz Ireneusz ya,M,30.0,185.0,96.0,Poland,POL,1998 Winter,1998,Winter,Nagano,Bobsleigh,Bobsleigh Men's Four,None


In [20]:
# 1- Cuantos Juegos Olimpicos se han realizado?

duckdb.query("""
select count(distinct year) 
as Total_juegos 
from athletes
""").to_df()

,Total_juegos
0,35


In [21]:
# 2- Enumerar todos los Juegos Olimpicos realizados hasta ahora

duckdb.query("""
select distinct games 
from athletes
ORDER BY games
""").to_df()

,Games
0,1896 Summer
1,1900 Summer
2,1904 Summer
3,1906 Summer
4,1908 Summer
5,1912 Summer
6,1920 Summer
7,1924 Summer
8,1924 Winter
9,1928 Summer


In [22]:
# 3- Numero total de naciones que participaron en cada Juego Olimpico

duckdb.query("""
select Games, count(distinct noc) 
as Num_Naciones
from athletes
group by Games
""").to_df()

,Games,Num_Naciones
0,2014 Winter,89
1,1936 Winter,28
2,2012 Summer,205
3,1994 Winter,67
4,1984 Winter,49
5,1900 Summer,31
6,1928 Summer,46
7,1932 Summer,47
8,1956 Summer,72
9,1988 Summer,159


In [23]:
# 4- Año en que se vio el mayor y menor numero de paises participando en los Juegos Olimpicos
duckdb.query("""
(select 'Maximo' as tipo, year, count(distinct noc) as numero_paises
from athletes
group by year
order by numero_paises desc
limit 1
)
union
(select 'Minimo' as tipo, year, count(distinct noc) as numero_paises
from athletes
group by year
order by numero_paises asc
limit 1
)
""").to_df()

,tipo,Year,numero_paises
0,Maximo,2016,207
1,Minimo,1896,12


In [24]:
# 5- Que nación ha participado en todos los Juegos Olimpicos? 

duckdb.query("""
select noc, count(distinct year) as Num_Participaciones
from athletes
group by noc
having Num_Participaciones =
(select count (distinct year)
from athletes)
""").to_df()

,NOC,Num_Participaciones
0,SUI,35
1,FRA,35
2,GRE,35
3,USA,35
4,ITA,35
5,GBR,35


In [25]:
# 6 - Identificar el deporte que se jugo en todas las olimpiadas de verano

duckdb.query("""
SELECT sport, COUNT(DISTINCT games) AS Num_juegos
FROM athletes
WHERE season = 'Summer'
GROUP BY sport
HAVING COUNT(DISTINCT games) = (
    SELECT COUNT(DISTINCT games)
    FROM athletes
    WHERE season = 'Summer'
)
""").to_df()

,Sport,Num_juegos
0,Athletics,29
1,Fencing,29
2,Swimming,29
3,Cycling,29
4,Gymnastics,29


In [26]:
# 7 - Identificar el deporte que se jugo en todas las olimpiadas de verano

duckdb.query("""
SELECT DISTINCT sport
FROM athletes
WHERE season = 'Summer'
GROUP BY sport
HAVING COUNT(DISTINCT year) = 
(SELECT COUNT(DISTINCT year) from athletes
WHERE season = 'Summer')
""").to_df()


,Sport
0,Gymnastics
1,Athletics
2,Swimming
3,Fencing
4,Cycling


In [27]:
# 8 - Obtener el numero total de deportes jugados en cada Juego Olimpico

duckdb.query("""
SELECT games, COUNT(DISTINCT sport) AS Total_Juegos
FROM athletes
GROUP BY games
""").to_df()

,Games,Total_Juegos
0,1988 Summer,27
1,1956 Summer,19
2,1972 Summer,23
3,2006 Winter,15
4,1936 Summer,24
5,2004 Summer,34
6,1980 Summer,23
7,1924 Winter,10
8,2010 Winter,15
9,1960 Winter,8


In [28]:
# 9 - Encontrar la proporcion de atletas masculinos y femeninos que participaron en todos los Juegos Olimpicos

duckdb.query("""
SELECT year, round( MaleCount/TotalCount, 2) AS Male_Proporcion, -- Obtenemos la proporcion de atletas masculinos y femeninos 
             round(FemaleCount/TotalCount, 2) AS Female_Proporcion 
From( 
SELECT year, COUNT(*) AS TotalCount,
sum(case when sex = 'M' then 1 else 0 end) *100 as MaleCount,
sum(case when sex = 'F' then 1 else 0 end) *100 as FemaleCount
from athletes
group by year) AS sq1
order by year
""").to_df()

,Year,Male_Proporcion,Female_Proporcion
0,1896,100.00,0.00
1,1900,98.30,1.70
2,1904,98.77,1.23
3,1906,99.37,0.63
4,1908,98.48,1.52
5,1912,97.85,2.15
6,1920,96.88,3.12
7,1924,95.42,4.58
8,1928,92.16,7.84
9,1932,88.89,11.11


In [29]:
# 10 - Buscar los 5 mejores atletas que han ganado la mayor cantidad de medallas de oro

duckdb.query("""
SELECT Name, COUNT(*) AS Total_medallas_oro
FROM athletes
WHERE Medal = 'Gold'
GROUP BY Name
ORDER BY Total_medallas_oro DESC
LIMIT 5
""").to_df()

,Name,Total_medallas_oro
0,"Michael Fred Phelps, II",23
1,"Raymond Clarence ""Ray"" Ewry",10
2,Paavo Johannes Nurmi,9
3,"Frederick Carlton ""Carl"" Lewis",9
4,Larysa Semenivna Latynina (Diriy-),9


In [30]:
# 11 - Obtener los 5 mejores atletas que han ganado la mayor cantidad de medallas (oro, plata y bronce)

duckdb.query("""
SELECT Name, 
sum(case when Medal = 'Gold' then 1 else 0 end) AS Medallas_oro,
sum(case when Medal = 'Silver' then 1 else 0 end) AS Medallas_plata,
sum(case when Medal = 'Bronze' then 1 else 0 end) AS Medallas_bronce,
             COUNT(*) AS Total_medallas
From athletes
             WHERE Medal IN ('Gold', 'Silver', 'Bronze')
GROUP BY Name
ORDER BY Total_medallas DESC
LIMIT 5
""").to_df()

,Name,Medallas_oro,Medallas_plata,Medallas_bronce,Total_medallas
0,"Michael Fred Phelps, II",23.0,3.0,2.0,28
1,Larysa Semenivna Latynina (Diriy-),9.0,5.0,4.0,18
2,Nikolay Yefimovich Andrianov,7.0,5.0,3.0,15
3,Edoardo Mangiarotti,6.0,5.0,2.0,13
4,Takashi Ono,5.0,4.0,4.0,13


In [31]:
# 12 - Obtener los 5 paises mas exitosos en los Juegos Olimpicos (El exito se mide por el numero de medallas ganadas)

duckdb.query("""
SELECT team AS Country,
sum(case when Medal = 'Gold' then 1 else 0 end) AS Medallas_oro,
sum(case when Medal = 'Silver' then 1 else 0 end) AS Medallas_plata,
sum(case when Medal = 'Bronze' then 1 else 0 end) AS Medallas_bronce,
COUNT(*) AS Total_medallas
FROM athletes
WHERE Medal IN ('Gold', 'Silver', 'Bronze')
GROUP BY team
ORDER BY Total_medallas DESC
LIMIT 5
""").to_df()

,Country,Medallas_oro,Medallas_plata,Medallas_bronce,Total_medallas
0,United States,2474.0,1512.0,1233.0,5219
1,Soviet Union,1058.0,716.0,677.0,2451
2,Germany,679.0,627.0,678.0,1984
3,Great Britain,519.0,582.0,572.0,1673
4,France,455.0,518.0,577.0,1550


In [32]:
# 13 - Enumerar el numero total de medallas de oro, plata y bronce ganadas en cada Pais

duckdb.query("""
SELECT team AS Country,
sum(case when Medal = 'Gold' then 1 else 0 end) AS Medallas_oro,
sum(case when Medal = 'Silver' then 1 else 0 end) AS Medallas_plata,
sum(case when Medal = 'Bronze' then 1 else 0 end) AS Medallas_bronce,
COUNT(*) AS Total_medallas
FROM athletes
WHERE Medal IN ('Gold', 'Silver', 'Bronze')
GROUP BY team
ORDER BY Country
""").to_df()

,Country,Medallas_oro,Medallas_plata,Medallas_bronce,Total_medallas
0,A North American Team,0.0,0.0,4.0,4
1,Afghanistan,0.0,0.0,2.0,2
2,Algeria,5.0,4.0,8.0,17
3,Ali-Baba II,0.0,0.0,5.0,5
4,Amateur Athletic Association,5.0,0.0,0.0,5
...,...,...,...,...,...
493,Winnipeg Shamrocks-1,12.0,0.0,0.0,12
494,Yugoslavia,130.0,167.0,93.0,390
495,Zambia,0.0,1.0,1.0,2
496,Zimbabwe,17.0,4.0,1.0,22


In [33]:
# 14 - Enumerar el numero total de medallas de oro, plata y bronce ganadas por cada Pais con relacion a cada Juego Olimpico

duckdb.query("""
SELECT team AS Country,
year AS Olimpic_year,
sum(case when Medal = 'Gold' then 1 else 0 end) AS Medallas_oro,
sum(case when Medal = 'Silver' then 1 else 0 end) AS Medallas_plata,
sum(case when Medal = 'Bronze' then 1 else 0 end) AS Medallas_bronce,
from athletes
WHERE Medal IN ('Gold', 'Silver', 'Bronze')  
GROUP BY team, year
ORDER BY Country, Olimpic_year
""").to_df()

,Country,Olimpic_year,Medallas_oro,Medallas_plata,Medallas_bronce
0,A North American Team,1900,0.0,0.0,4.0
1,Afghanistan,2008,0.0,0.0,1.0
2,Afghanistan,2012,0.0,0.0,1.0
3,Algeria,1984,0.0,0.0,2.0
4,Algeria,1992,1.0,0.0,1.0
...,...,...,...,...,...
2007,Zambia,1996,0.0,1.0,0.0
2008,Zimbabwe,1980,15.0,0.0,0.0
2009,Zimbabwe,2004,1.0,1.0,1.0
2010,Zimbabwe,2008,1.0,3.0,0.0


In [34]:
# 15 - Indentificar que pais gano la mayoria de las medallas de oro, plata, y bronce en cada Juego Olimpico

duckdb.query("""
SELECT year AS Olympic_year,
Medal AS Medal_Type,
team AS Winning_Country,
Medal_Count
FROM (
    SELECT year,
           team,
           Medal,
           COUNT(*) AS Medal_Count,
           RANK() OVER (
               PARTITION BY year, Medal
               ORDER BY COUNT(*) DESC
           ) AS Rank
    FROM athletes
    WHERE Medal IN ('Gold','Silver','Bronze')
    GROUP BY year, team, Medal
) ranked
WHERE Rank = 1
ORDER BY Olympic_year, Medal_Type
""").to_df()

,Olympic_year,Medal_Type,Winning_Country,Medal_Count
0,1896,Bronze,Greece,18
1,1896,Gold,Germany,24
2,1896,Silver,Greece,16
3,1900,Bronze,France,26
4,1900,Gold,France,22
...,...,...,...,...
105,2016,Bronze,Germany,67
106,2016,Bronze,United States,67
107,2016,Gold,United States,137
108,2016,Silver,France,55


In [35]:
# 16 - En que deportes/eventos India ha ganado la mayor cantidad de medallas

duckdb.query("""
SELECT Sport, event,
             COUNT(*) AS Total_medallas,
             sum(case when Medal = 'Gold' then 1 else 0 end) AS Medallas_oro,
             sum(case when Medal = 'Silver' then 1 else 0 end) AS Medallas_plata,
             sum(case when Medal = 'Bronze' then 1 else 0 end) AS Medallas_bronce,
From athletes
             WHERE team = 'Argentina' and Medal IN ('Gold', 'Silver', 'Bronze')
             GROUP BY Sport, event
             ORDER BY Medallas_oro DESC, medallas_plata DESC, medallas_bronce DESC

""").to_df()

,Sport,Event,Total_medallas,Medallas_oro,Medallas_plata,Medallas_bronce
0,Football,Football Men's Football,68,34.0,34.0,0.0
1,Hockey,Hockey Men's Hockey,18,18.0,0.0,0.0
2,Basketball,Basketball Men's Basketball,24,12.0,0.0,12.0
3,Polo,Polo Men's Polo,9,9.0,0.0,0.0
4,Boxing,Boxing Men's Heavyweight,5,3.0,1.0,1.0
5,Boxing,Boxing Men's Featherweight,5,2.0,1.0,2.0
6,Athletics,Athletics Men's Marathon,3,2.0,1.0,0.0
7,Sailing,Sailing Mixed Multihull,6,2.0,0.0,4.0
8,Cycling,Cycling Men's Madison,2,2.0,0.0,0.0
9,Rowing,Rowing Men's Double Sculls,2,2.0,0.0,0.0


In [36]:
# 17 - Desglosar todos los Juegos Olimpicos en los que India gano medallas en hockey y cuantas medallas gano en cada uno de ellos

duckdb.query("""
SELECT year as Olympic_year,
             count(*) as Total_Medallas,
             sum(case when Medal = 'Gold' then 1 else 0 end) AS Medallas_oro,
             sum(case when Medal = 'Silver' then 1 else 0 end) AS Medallas_plata,
             sum(case when Medal = 'Bronze' then 1 else 0 end) AS Medallas_bronce,
From athletes
             WHERE team = 'Argentina' and Medal IN ('Gold', 'Silver', 'Bronze') and Sport = 'Football'
             GROUP BY year
             ORDER BY Olympic_year

""").to_df()

,Olympic_year,Total_Medallas,Medallas_oro,Medallas_plata,Medallas_bronce
0,1928,16,0.0,16.0,0.0
1,1996,18,0.0,18.0,0.0
2,2004,16,16.0,0.0,0.0
3,2008,18,18.0,0.0,0.0
